# Clase 3 — Relevar un dataset

**Laboratorio de Modelo Analítico para la Toma de Decisiones**
ISTEA · 2026-2C · División A

---

**Relevar** es entender qué datos hay antes de tocarlos: qué representa cada fila,
qué significa cada columna, de dónde salió y si alcanza para responder la pregunta.

Es la parte menos glamorosa del oficio y la que más proyectos salva. Un modelo
espectacular sobre un dato mal entendido no sirve para nada.

### Cómo está armado este notebook

| Parte | Qué es | Cuándo |
| --- | --- | --- |
| **A** | Relevamiento de ElecCommSphere, el caso de clase | Lo hacemos juntos |
| **B** | Las cuatro tareas sobre **tu** dataset | Segunda hora, en equipo |

Lo que produzcas en la parte B es directamente el primer pedazo del **TP1**. No es
un ejercicio para tirar.

---
# PARTE A · Relevamiento de ElecCommSphere

Seguí esta parte mirando. No hace falta que la corras, aunque podés.

In [ ]:
import pandas as pd

tx = pd.read_csv("eleccommsphere_transacciones.csv", parse_dates=["fecha"])
print("Filas y columnas:", tx.shape)

## A1 · La pregunta que va primero: ¿qué es una fila?

A eso se le llama la **granularidad** de la tabla.

> Tres nombres para lo mismo, según quién lo diga: **granularidad** en el mundo de
> datos, **unidad de análisis** en estadística, y **grain** en la documentación en
> inglés, que es casi toda. Conviene reconocer los tres.

Parece obvio y casi nunca lo es. Una tabla de e-commerce puede tener como unidad:

- una compra
- un producto dentro de una compra
- un cliente
- un cliente por mes

Y según cuál sea, contar filas significa cosas completamente distintas. **La mitad de
los análisis mal hechos empiezan acá**: alguien contó filas creyendo que contaba
personas.

### Cómo se determina

Buscás la columna que creés que identifica cada fila y contás cuántos valores
distintos tiene. Si ese número coincide con la cantidad de filas, esa es la
granularidad.

In [ ]:
print("Filas totales:            ", len(tx))
print("id_transaccion distintos: ", tx["id_transaccion"].nunique())
print("id_cliente distintos:     ", tx["id_cliente"].nunique())

### Y acá aparece la primera sorpresa

Hay **26.310 filas** pero solo **26.130 id_transaccion distintos**. O sea que hay 180
filas que comparten identificador con otra.

Eso puede significar dos cosas muy distintas, y hay que averiguar cuál:

1. **La granularidad es más fina de lo que pensábamos.** Por ejemplo, si la tabla
   tuviera una fila por producto dentro de cada compra, el id de la compra se
   repetiría legítimamente.
2. **Hay un problema de calidad.** Filas cargadas dos veces, o peor: el mismo
   identificador usado para dos cosas distintas.

Se distinguen mirando si las filas repetidas son idénticas o no.

In [ ]:
repetidas = tx[tx["id_transaccion"].duplicated(keep=False)]

# Para cada id repetido: ¿cuántas versiones DISTINTAS de esa fila hay?
versiones = repetidas.groupby("id_transaccion").apply(
    lambda d: d.drop_duplicates().shape[0], include_groups=False
)

print("Ids que aparecen más de una vez :", repetidas["id_transaccion"].nunique())
print("  con filas idénticas           :", (versiones == 1).sum(), " -> carga duplicada")
print("  con filas distintas           :", (versiones > 1).sum(), " -> problema de integridad")

### Lo que significa

**179 son filas idénticas**: la misma compra cargada dos veces. Es el clásico error
de un proceso que corrió dos veces. Molesto, fácil de arreglar.

**Pero hay 1 que no.** Un `id_transaccion` que aparece dos veces con datos
diferentes. Ese es otro tipo de problema y es más grave. Miremos cuál es.

In [ ]:
conflictivo = versiones[versiones > 1].index[0]
tx[tx["id_transaccion"] == conflictivo]

### Mismo identificador, dos clientes distintos

La transacción `99` figura una vez a nombre del cliente `373` y otra a nombre del
`999999`. Todo lo demás —fecha, canal, monto, categoría— es igual.

¿Cuál de los dos compró? **No hay forma de saberlo desde el archivo.** Eso se
resuelve preguntándole a Sistemas, no adivinando. Y si nadie puede responder, la
decisión de qué hacer con esa fila hay que escribirla y justificarla.

> Fijate además en el `999999`. Ese número tiene pinta de comodín, de esos que un
> sistema usa cuando no sabe qué poner. Anotalo: la clase que viene vamos a ver que
> ese cliente **no existe** en la tabla de clientes.

### Entonces, ¿cuál es la granularidad?

Cada fila es una compra. Los 180 casos no son una granularidad más fina: son
suciedad. Pero eso lo sabemos **porque lo verificamos**, no porque lo supusimos.

> **La granularidad de ElecCommSphere, escrita en una oración:**
> *Cada fila es una compra realizada por un cliente en una fecha, a través de un
> canal, de un producto de una categoría — con 180 filas duplicadas que hay que
> resolver antes de contar nada.*

Esa oración va en el TP1. Escribirla obliga a entender la tabla, y la segunda parte
solo aparece si uno se tomó el trabajo de mirar.

## A2 · Qué tipo tiene cada columna: `info()`

`info()` da una línea por columna con tres datos: cómo se llama, cuántos valores
**no nulos** tiene y de qué tipo la leyó pandas.

In [ ]:
tx.info()

### Cómo se lee esto

**Mirá la columna del medio, la de «non-null».** Si dice un número menor que el total
de filas, ahí hay agujeros. Cuántos faltan es la resta.

**Mirá los tipos.** `object` casi siempre quiere decir texto. Si una columna que
debería ser numérica aparece como `object`, hay basura adentro: un «N/A» escrito a
mano, un símbolo de peso, una coma en vez de punto.

En esta tabla hay dos columnas con nulos. Todavía no las tocamos — hoy solo las
anotamos.

## A3 · Órdenes de magnitud: `describe()`

`describe()` solo mira las columnas numéricas y devuelve ocho números por columna.
No hace falta leerlos todos: hay tres que importan.

In [ ]:
tx[["monto", "descuento_pct", "busquedas_sesion"]].describe().round(1)

### Los tres que importan

**El mínimo y el máximo.** Es donde aparecen los errores de carga. Un monto mínimo
negativo no es un cliente raro: es un error, o una devolución mal registrada.

**La media contra el 50 % (la mediana).** Si son muy distintas, la distribución tiene
cola larga y el promedio miente. Para describir al cliente típico, en ese caso, se
usa la mediana.

**El desvío (`std`) comparado con la media.** Si el desvío es más grande que la
media, hay valores extremos. Acá el desvío del monto es casi nueve veces la media:
eso ya anticipa outliers antes de graficar nada.

In [ ]:
# Un chequeo rápido de cosas que no deberían pasar
print("Montos negativos o cero:", (tx["monto"] <= 0).sum())
print("Montos nulos:           ", tx["monto"].isna().sum())
print("Filas duplicadas:       ", tx.duplicated().sum())
print()
print("Categorías escritas de formas distintas:", tx["categoria"].nunique(), "valores")
print(sorted(tx["categoria"].unique())[:8], "...")

Cinco rubros reales, escritos de diecisiete formas distintas. Eso es un problema de
calidad y lo vamos a arreglar la clase que viene.

> **Hoy no limpiamos nada.** Esta pasada es para poder escribir el diccionario y
> saber qué preguntar. La tentación de limpiar antes de entender es enorme y produce
> decisiones que después no se pueden justificar.

## A4 · El diccionario de datos

Un **diccionario de datos** es la ficha técnica de la tabla. Cinco casillas por
columna:

| Casilla | Qué se pone |
| --- | --- |
| Campo | El nombre exacto, como está en el archivo |
| Tipo | int, float, fecha, texto, categoría, booleano |
| Descripción | Qué representa. **Con unidad**: pesos, días, porcentaje |
| Origen | De qué sistema sale. Si no sabés, «a confirmar» |
| ¿Sirve? | Si aporta al problema, y por qué |

### Cuatro formas de arruinarlo

1. **Decir «Sí» en todo.** Si todas las columnas sirven, no lo pensaste. Un
   diccionario honesto marca al menos una como «no aporta» y explica por qué.
2. **Describir el nombre.** «fecha: la fecha» no es una descripción. Sí lo es:
   «fecha en que se confirmó la compra, registrada por el e-commerce».
3. **No decir la unidad.** ¿Pesos o dólares? ¿El monto incluye envío e impuestos?
4. **Inventar el origen.** «A confirmar» es información honesta y vale más que una
   suposición disfrazada de dato.

El diccionario completo de ElecCommSphere está en la **sección 1.1 del notebook
general**. Usalo de molde.

---
# PARTE B · Ahora, tu dataset

De acá en adelante trabajan en equipo. Son **cuatro tareas** y las cuatro entran
después en el TP1.

> **Antes de arrancar:** carguen su archivo. El bloque de carga de los trece datasets
> está en el notebook de la clase 2, celda 6. Copien las líneas del suyo acá abajo.

In [ ]:
import pandas as pd

# ---- PEGÁ ACÁ LAS LÍNEAS DE CARGA DE TU DATASET ----
# (están en el notebook de la clase 2, celda 6)

# ---- Y COMPLETÁ ESTOS TRES NOMBRES ----
ID    = ""      # la columna que identifica al cliente
FECHA = ""      # la columna de fecha
VALOR = ""      # la columna del valor sumable

print("Cargado:", tx.shape[0], "filas y", tx.shape[1], "columnas")

## Tarea 1 · Determinar la granularidad

¿Qué es una fila en tu dataset? Probá con la columna que creas que la identifica.

Si ninguna columna sola identifica la fila, puede que la granularidad sea una
**combinación** de dos: por ejemplo, orden + producto. Eso también hay que escribirlo.

In [ ]:
# Cambiá el nombre por la columna que creas que identifica cada fila
CANDIDATA = "id_transaccion"

n, u = len(tx), tx[CANDIDATA].nunique()
print(f"Filas totales      : {n:,}")
print(f"{CANDIDATA} distintos: {u:,}")
print()

if u == n:
    print("GRANULARIDAD ENCONTRADA: cada fila es un", CANDIDATA)
else:
    print(f"Esa columna se repite en {n-u:,} filas. Puede ser por dos motivos:")
    rep = tx[tx[CANDIDATA].duplicated(keep=False)]
    vers = rep.groupby(CANDIDATA).apply(lambda d: d.drop_duplicates().shape[0],
                                        include_groups=False)
    print(f"   filas idénticas   : {(vers == 1).sum():,}  -> carga duplicada (calidad)")
    print(f"   filas distintas   : {(vers > 1).sum():,}  -> granularidad más fina, o integridad")
    print()
    print("Si casi todas son idénticas, la granularidad es esta y hay suciedad.")
    print("Si casi todas son distintas, es más fina: probá combinando dos columnas.")

> ### Para escribir
> La granularidad de nuestro dataset en una oración:
>
> *Cada fila es . . . . . . . . . . . . . . . . . . . . . . . . . . . . .*
>
> Si les da distinto de lo que esperaban, mejor: eso es un hallazgo y va en el TP1.
> Pasa seguido — en varios de los datasets del catálogo la fila no es lo que parece.

## Tarea 2 · Verificar la ficha del catálogo

En la hoja de su equipo hay cuatro números: filas, identificadores únicos, filas por
identificador y rango de fechas.

Los medí abriendo el archivo, no leyendo la web. Pero medí **la tabla principal, no
todas**, y medí **la forma, no el contenido**. Así que puede haber diferencias.

Reprodúzcanlos.

In [ ]:
f = pd.to_datetime(tx[FECHA], errors="coerce")
n_id = tx[ID].nunique()

print("=== Lo que dice el archivo ===")
print(f"Filas             : {len(tx):,}")
print(f"Ids únicos        : {n_id:,}")
print(f"Filas por id      : {len(tx)/n_id:.1f}")
print(f"Rango de fechas   : {f.min().date()}  a  {f.max().date()}")
print(f"Meses cubiertos   : {(f.max()-f.min()).days/30.44:.0f}")
print()
print("Compará con tu ficha. ¿Coincide todo?")

> ### Para anotar
> - ¿Coinciden los cuatro números? **Sí / No**
> - Si alguno no coincide, ¿cuál y por cuánto?
> - ¿Encontraron algo que la ficha no menciona?
>
> Si hay diferencias, avísenme por mail: corrijo el catálogo y les suma en el TP1.
> Un relevamiento que confirma la documentación sin abrir el archivo no es un
> relevamiento.

## Tarea 3 · Primera lectura de calidad

Todavía **no se arregla nada**. Solo se cuenta y se anota. Arreglar es la clase 4.

In [ ]:
print("=== Forma y tipos ===")
tx.info()

In [ ]:
print("=== Nulos por columna (solo las que tienen) ===")
nulos = tx.isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
if len(nulos) == 0:
    print("Ninguna columna tiene nulos.")
else:
    for col, n in nulos.items():
        print(f"  {col:30s} {n:>9,}  ({n/len(tx):.1%})")

print()
print("=== Otros chequeos ===")
print("Filas duplicadas exactas:", tx.duplicated().sum())

# Si tu VALOR es plata o cantidad, esto tiene sentido.
# Si es un puntaje de 1 a 5, saltealo.
print("Valores <= 0 en", VALOR + ":", (pd.to_numeric(tx[VALOR], errors="coerce") <= 0).sum())

In [ ]:
# Rangos: acá aparecen los errores de carga
tx.describe().round(2)

> ### Para anotar
> Tres problemas de calidad que encontraron, con el número al lado:
>
> 1. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .
> 2. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .
> 3. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .
>
> **Con números, no con adjetivos.** «Hay bastantes nulos» no sirve. «La columna X
> tiene 4.382 nulos, el 16 % de las filas» sí.

## Tarea 4 · Arrancar el diccionario

Las primeras **seis columnas** de su tabla principal. Las demás quedan de tarea.

La celda de abajo les imprime el esqueleto para que no arranquen de una hoja en
blanco: cada columna con su tipo, cuántos nulos tiene y algunos valores de ejemplo.
Lo que falta —descripción, origen y si sirve— lo escriben ustedes.

In [ ]:
print("ESQUELETO DEL DICCIONARIO — completá las tres últimas casillas a mano\n")
print(f"{'CAMPO':28s} {'TIPO':12s} {'NULOS':>8s}   EJEMPLOS")
print("-" * 100)

for col in tx.columns[:6]:
    tipo   = str(tx[col].dtype)
    nulos  = tx[col].isna().sum()
    ejemplos = tx[col].dropna().unique()[:3]
    ejemplos = ", ".join(str(e)[:18] for e in ejemplos)
    print(f"{col[:28]:28s} {tipo:12s} {nulos:>8,}   {ejemplos}")

print()
print("Para cada una, completar:")
print("   Descripción  -> qué representa, CON la unidad")
print("   Origen       -> de qué sistema sale (o «a confirmar»)")
print("   ¿Sirve?      -> y por qué. Al menos una tiene que decir «no aporta»")

> ### Recordá
> Al menos **una** columna tiene que estar marcada como «no aporta», con su motivo.
> Si todas sirven, no lo pensaron.

---

## Lo que se llevan de hoy

| Producto | Dónde va |
| --- | --- |
| La granularidad, en una oración | TP1 — sección de relevamiento |
| La verificación de la ficha | TP1 — y me avisan si hay diferencias |
| Tres problemas de calidad con números | TP1 — informe de calidad |
| El diccionario de seis columnas | TP1 — se completa esta semana |

## Para la clase 4

1. Terminar el diccionario de **todas** las columnas.
2. Traer el diagrama del ecosistema **en borrador** — lo corregimos en clase, y esa
   devolución vale más que cualquier cosa que puedan leer.
3. Anotar **tres columnas que no entienden**. Las vamos a usar de ejemplo.

## Si se traban

Escriban por mail. Para que la respuesta sea útil, la pregunta tiene que tener cuatro
cosas: qué intentaron, qué esperaban que pasara, qué pasó en realidad, y el mensaje
de error completo copiado y pegado.

Una pregunta así se responde en cinco minutos. «No me anda» no se responde nunca.